In [17]:
import sys
import os

print("PYTHON EXECUTABLE:")
print(sys.executable)

print("\nPYTHON VERSION:")
print(sys.version)

print("\nCURRENT DIRECTORY:")
print(os.getcwd())

PYTHON EXECUTABLE:
c:\Users\91975\anaconda3\envs\agentic\python.exe

PYTHON VERSION:
3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]

CURRENT DIRECTORY:
c:\Users\91975\OneDrive - Graphic Era University\Desktop\Agentic AI\research


In [18]:
import os 
import certifi 
from dotenv import load_dotenv

load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langchainhub import Client
from langchain.tools import tool
import requests

In [19]:
from langchain.agents import create_agent

In [20]:
os.getenv("GOOGLE_API_KEY")
os.getenv("TAVILY_API_KEY")
os.getenv("WEATHERSTACK_API_KEY")
print(os.getenv("WEATHERSTACK_API_KEY"))

a699c34bcaac516ef694f0259e397907


In [21]:
search_tool = TavilySearch(
    max_results=2
)

In [22]:
response= search_tool.invoke("What is the latest political news in india?")
response

{'query': 'What is the latest political news in india?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.news18.com/politics',
   'title': 'Politics News Live, Latest Indian Political News Today - News18',
   'content': '### \'BJP Will Win Datia Seat In 2028 MP Assembly Election\': CM Mohan Yadav At News18 Diamond States Summit\n\n## Latest politics News\n\nDK Shivakumar inducted 19 ministers into the Karnataka cabinet. Here\'s the full list of leaders sworn in, those left out, the Gayathri Shantegowda controversy and Congress dissent. (IMAGE: PTI FILE)")\n\n### Nobody Has Resigned After Karnataka Cabinet Rejig, Will Accommodate Seniors, Women: State Cong Chief | Exclusive [...] ### \'No Room For Ambiguity\': How A Delhi Lunch Served A Stern Message From BJP To NCPI\n\nAnand Ranganathan\'s Explosive Take on CJP: "Hypocrites Who Won" | AAP Link Row | News18\n\n### Anand Ranganathan\'s Explosive Take on CJP: "Hypocrites Who Won" | AAP Link 

In [23]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={os.getenv('WEATHERSTACK_API_KEY')}&query={city}"
    )
    
    response=requests.get(url)
    
    data=response.json()
    
    if "current" not in data:
        return f"Error: Unable to fetch weather data for {city}"
    
    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )
    
    
    
    

In [24]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.5,
    google_api_key=os.getenv("GOOGLE_API_KEY"))

In [25]:
response = llm.invoke("Tell me a joke about AI")
response


AIMessage(content=[{'type': 'text', 'text': 'How many AIs does it take to change a lightbulb?\n\n"As an AI, I cannot physically change a lightbulb. However, I can generate a 1,000-word essay on the history of electricity, followed by a Python script that simulates a lightbulb turning on."', 'extras': {'signature': 'EsgWCsUWARFNMg9+GL6yfHn0GeSmtBnQNPENDkxQlz9GUu0qmwN5P4E8hrwxuIFKFewtmwhu2MugiZruBygsN1hOvDdFNoFYFdDmovmh28Z0VmYGFq1m5z+jguNJsxEa24ocoU/ByQJfcbkHXnJkeVhAPxsniQ7lqGf1UKOToSiwewD3JM8HlSD6uiwKzfdm1HymUPL4Tm50RuUWLknb55QSDRUMxvrb+l7cRJWg6OEKXCLGjz1AIpbTRsVIZ0REIbxl6aYC9ZzG7t/VJhoYYi+/+A4J0Z50m5bShDoS2xgNG5u/U27jxbxJNK/qYX4MaGrHLSeYWT/cGHCOGWxu9kq6yTQ/13In4r20PtnASs+QdHxYSNv1wZQnaNB+JtXTQd6NJo47H1X61Hl12YASf8KVgtQ5tQ04kxvXb1ycwXdd4fmgvSGhQXKQONhc4VntfdxTxkOF0NYITLemiwLyqyB1/N2xrKQDoFZe6ZBGX4+VmgmhDrOtlF473kKcAz7muOYIlIstakPmNkr9nnyyyH54KXg5xaZsjlLo1Q3pOaAUAGrvd1r7THXanzriR12rMnOdBRB+vTV8TlmeDke7q4drIMRQL3jT1rTEeZk0AEjbA0yKPYgQw3sjue8VKvtmy5ZaaDBfpBiZ1PuWOXeiybaXxAlZm2IVclzj3I5y5Y+

In [26]:
# To know what models my api could use

# from google import genai
# import os

# client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# for model in client.models.list():
#     print(model.name)

In [27]:
# hub=Client()
# prompt=hub.pull("hwchase17/react")    #It was a pre-built ReAct prompt

'''Purane ke saath use hota tha , can't use with new version, it expects system instruction not old prompt template'''


prompt = """
You are a helpful assistant.
Use the available tools when necessary to answer the user's question.
"""

In [28]:
tools=[search_tool, get_weather_data]

In [29]:
agent=create_agent(model=llm, tools=tools, system_prompt=prompt)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the capital of India and then find its current weather?"
        }
    ]
})

print(result["messages"][-1].content)

[{'type': 'text', 'text': 'The capital of India is **New Delhi**. \n\nThe current weather in New Delhi is:\n* **Temperature:** 31°C\n* **Condition:** Smoky haze\n* **Humidity:** 64%', 'extras': {'signature': 'EuQBCuEBARFNMg+Aety5nmaDBeswEfXQ1zrbQ3WXPlQXDN87NW+dNidhxyRxROSGz23oZXrVaXWnyPim8NfH1YQJldTvmwcHaPelWmEHMPu7lW0YOrhei6pcW0Odr7+vVpwluM16GZIEs1P0J7A/Fqu6zpw/ZQLK7gwQfqPGZzVqs/G2rHGvmxoJagNHBSUSmWNq+zmQ53QjebeHh1iugIm9qegwSzAIFkKc0o9hQDYa3GAs7SUGAbYs3pMnEwVhq1GF/MdnrxdfjTXEjtkeIqcZ1uA4JjQRaAs8F8H4FoztVOfFjALy'}}]


In [30]:
# agent_executor = agent.as_executor(agent_name="Agentic AI", verbose=True, tools=tools)
'''New LangChain me AgentExecutor nhi chahiye hota'''

'New LangChain me AgentExecutor nhi chahiye hota'